In [13]:
import os
import random
import itertools
import numpy as np
import pandas as pd


def get_k_grams(text, k, type='char'):
    if type == 'word':
        tokens = text.split()
        return set([" ".join(tokens[i:i+k]) for i in range(len(tokens)-k+1)])
    else:
        return set([text[i:i+k] for i in range(len(text)-k+1)])

def jaccard_similarity(set1, set2):
    if not set1 or not set2: return 0
    return len(set1.intersection(set2)) / len(set1.union(set2))

def build_minhash_signatures(shingle_sets, t, m=10007):
    hash_params = [(random.randint(1, m-1), random.randint(0, m-1)) for _ in range(t)]
    all_shingles = list(set().union(*shingle_sets))
    shingle_to_id = {s: i for i, s in enumerate(all_shingles)}
    
    signatures = []
    for s_set in shingle_sets:
        shingle_ids = [shingle_to_id[s] for s in s_set]
        sig = []
        for a, b in hash_params:
            if not shingle_ids:
                sig.append(m)
            else:
                min_h = min([(a * r + b) % m for r in shingle_ids])
                sig.append(min_h)
        signatures.append(sig)
    return np.array(signatures)

# Load Data 
doc_names = ["D1", "D2", "D3", "D4"]
docs = {}
for name in doc_names:
    with open(f"input/{name}.txt", "r") as f:
        docs[name] = f.read().lower().strip()

In [14]:
print("TASK 1B: Exact Jaccard Similarities ")
gram_configs = [ (2, 'char'), (3, 'char'), (2, 'word') ]
doc_pairs = list(itertools.combinations(doc_names, 2))

for k, g_type in gram_configs:
    print(f"\nType: {k}-{g_type} grams")
    for d1, d2 in doc_pairs:
        s1 = get_k_grams(docs[d1], k, g_type)
        s2 = get_k_grams(docs[d2], k, g_type)
        sim = jaccard_similarity(s1, s2)
        print(f"J({d1}, {d2}): {sim:.4f}")

TASK 1B: Exact Jaccard Similarities 

Type: 2-char grams
J(D1, D2): 0.9811
J(D1, D3): 0.8157
J(D1, D4): 0.6444
J(D2, D3): 0.8000
J(D2, D4): 0.6413
J(D3, D4): 0.6530

Type: 3-char grams
J(D1, D2): 0.9780
J(D1, D3): 0.5804
J(D1, D4): 0.3051
J(D2, D3): 0.5680
J(D2, D4): 0.3059
J(D3, D4): 0.3121

Type: 2-word grams
J(D1, D2): 0.9408
J(D1, D3): 0.1823
J(D1, D4): 0.0302
J(D2, D3): 0.1737
J(D2, D4): 0.0303
J(D3, D4): 0.0161


In [15]:
print("\nTASK 2A: Min-Hash Approximation (D1 & D2) ")
t_values = [20, 60, 150, 300, 600]
d1_3char = get_k_grams(docs["D1"], 3, 'char')
d2_3char = get_k_grams(docs["D2"], 3, 'char')

for t in t_values:
    sigs = build_minhash_signatures([d1_3char, d2_3char], t)
    approx_sim = np.mean(sigs[0] == sigs[1])
    print(f"t={t:3}, Approx Jaccard: {approx_sim:.4f}")


TASK 2A: Min-Hash Approximation (D1 & D2) 
t= 20, Approx Jaccard: 0.9500
t= 60, Approx Jaccard: 0.9500
t=150, Approx Jaccard: 0.9933
t=300, Approx Jaccard: 0.9733
t=600, Approx Jaccard: 0.9800


In [16]:
print("\nTASK 3B: LSH Probabilities (t=160, b=20, r=8)")
b, r = 20, 8
for d1, d2 in doc_pairs:
    s1 = get_k_grams(docs[d1], 3, 'char')
    s2 = get_k_grams(docs[d2], 3, 'char')
    s = jaccard_similarity(s1, s2)
    prob = 1 - (1 - s**r)**b
    print(f"P(candidate {d1}-{d2}) at s={s:.4f}: {prob:.4f}")


TASK 3B: LSH Probabilities (t=160, b=20, r=8)
P(candidate D1-D2) at s=0.9780: 1.0000
P(candidate D1-D3) at s=0.5804: 0.2282
P(candidate D1-D4) at s=0.3051: 0.0015
P(candidate D2-D3) at s=0.5680: 0.1959
P(candidate D2-D4) at s=0.3059: 0.0015
P(candidate D3-D4) at s=0.3121: 0.0018


In [17]:

def task_4_minhash():
    print("\n TASK 4: Min-Hashing (Full Signature Comparison)")
    t_values = [50, 100, 200]
    target_sim = 0.5
    
    # 1. Get Exact Pairs (Ground Truth)
    exact_pairs = set()
    for i in range(len(user_ids)):
        for j in range(i + 1, len(user_ids)):
            if jaccard_similarity(user_sets[i], user_sets[j]) >= target_sim:
                exact_pairs.add((user_ids[i], user_ids[j]))
    
    # 2. Run Experiments for each t
    for t in t_values:
        fp_runs, fn_runs = [], []
        for run in range(5):
            sigs = build_minhash_signatures(user_sets, t)
            approx_pairs = set()
            
            # Nested loop: Comparing every signature pair
            for i in range(len(user_ids)):
                for j in range(i + 1, len(user_ids)):
                    # Similarity is the fraction of matching rows
                    approx = np.mean(sigs[i] == sigs[j])
                    if approx >= target_sim:
                        approx_pairs.add((user_ids[i], user_ids[j]))
            
            fp = len(approx_pairs - exact_pairs)
            fn = len(exact_pairs - approx_pairs)
            fp_runs.append(fp)
            fn_runs.append(fn)
            
        print(f"t={t:3} | Avg FP: {np.mean(fp_runs):.2f}, Avg FN: {np.mean(fn_runs):.2f}")

# Execute
task_4_minhash()


 TASK 4: Min-Hashing (Full Signature Comparison)
t= 50 | Avg FP: 155.80, Avg FN: 1.80
t=100 | Avg FP: 31.60, Avg FN: 2.20
t=200 | Avg FP: 10.60, Avg FN: 2.40


In [20]:
from collections import defaultdict
import itertools

def run_task_5():
    # Parameters provided in the prompt
    experiments = [
        {'r': 5, 'b': 10, 't': 50},
        {'r': 5, 'b': 20, 't': 100},
        {'r': 5, 'b': 40, 't': 200},
        {'r': 10, 'b': 20, 't': 200}
    ]
    
    thresholds = [0.6, 0.8]
    
    for target_sim in thresholds:
        print(f"\nEXPERIMENT: Target Similarity >= {target_sim}")

        # 1. Get Ground Truth (Exact Jaccard)
        exact_pairs = set()
        for i in range(len(user_ids)):
            for j in range(i + 1, len(user_ids)):
                if jaccard_similarity(user_sets[i], user_sets[j]) >= target_sim:
                    exact_pairs.add((user_ids[i], user_ids[j]))
        
        for exp in experiments:
            r, b, t = exp['r'], exp['b'], exp['t']
            fp_results, fn_results = [], []
            
            for run in range(5):
                # Generate signatures
                sigs = build_minhash_signatures(user_sets, t)
                candidate_pairs = set()
                
                # 2. LSH Banding
                for band_idx in range(b):
                    buckets = defaultdict(list)
                    for u_idx in range(len(user_ids)):
                        band = tuple(sigs[u_idx][band_idx * r : (band_idx + 1) * r])
                        buckets[hash(band)].append(u_idx)
                    
                    for bucket_users in buckets.values():
                        if len(bucket_users) > 1:
                            for p1, p2 in itertools.combinations(bucket_users, 2):
                                # Sort to ensure (u1, u2) is same as (u2, u1)
                                u1, u2 = sorted((user_ids[p1], user_ids[p2]))
                                candidate_pairs.add((u1, u2))
                
                # 3. Calculate FP and FN
                # False Positive: In candidates but actual Jaccard < target_sim
                fp = 0
                for u1, u2 in candidate_pairs:
                    if (u1, u2) not in exact_pairs:
                        fp += 1
                
                # False Negative: In exact_pairs but NOT in candidates
                fn = 0
                for pair in exact_pairs:
                    if pair not in candidate_pairs:
                        fn += 1
                
                fp_results.append(fp)
                fn_results.append(fn)
            
            print(f"Params: (r={r}, b={b}, t={t}) | Avg FP: {np.mean(fp_results):.1f} | Avg FN: {np.mean(fn_results):.1f}")

run_task_5()


EXPERIMENT: Target Similarity >= 0.6
Params: (r=5, b=10, t=50) | Avg FP: 641.4 | Avg FN: 0.4
Params: (r=5, b=20, t=100) | Avg FP: 1243.6 | Avg FN: 0.4
Params: (r=5, b=40, t=200) | Avg FP: 2453.2 | Avg FN: 0.0
Params: (r=10, b=20, t=200) | Avg FP: 4.6 | Avg FN: 1.2

EXPERIMENT: Target Similarity >= 0.8
Params: (r=5, b=10, t=50) | Avg FP: 619.6 | Avg FN: 0.0
Params: (r=5, b=20, t=100) | Avg FP: 1121.0 | Avg FN: 0.0
Params: (r=5, b=40, t=200) | Avg FP: 2667.4 | Avg FN: 0.0
Params: (r=10, b=20, t=200) | Avg FP: 8.8 | Avg FN: 0.0
